# Sequential estimation
Adapted from https://sbi-dev.github.io/sbi/latest/tutorials/02_multiround_inference/ for the two moons problem (https://arxiv.org/pdf/1905.07488 A.5.1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy import stats

from sbi.analysis import pairplot
from sbi.inference import NPE, simulate_for_sbi
from sbi.inference.trainers.npe.npe_d import NPE_D
from sbi.inference import NPE_B, NPE_C
from sbi.utils import BoxUniform
from sbi.utils.user_input_checks import (
    check_sbi_inputs,
    process_prior,
    process_simulator,
)

In [ ]:
def two_moons(theta):
    """Simulate two-moons data."""
    # From https://arxiv.org/pdf/1905.07488 A.5.1
    a = (torch.rand(1) - 0.5) * torch.pi
    r = torch.randn(1) * 0.01 + 0.1
    p = torch.cat([r * torch.cos(a) + 0.25, r * torch.sin(a)])
    x = p + torch.tensor([- torch.abs(theta[0] + theta[1]),
                          -theta[0] + theta[1]]
                        ) / np.sqrt(2)
    return x


def two_moons_likelihood(theta, x):
    """Ground truth likelihood for the two-moons simulator."""
    p = x - np.array([- np.abs(theta[0] + theta[1]),
                      -theta[0] + theta[1]]
                    ) / np.sqrt(2)
    p0x = p[0] - 0.25
    p0y = p[1]
    a = np.arctan2(p0y, p0x)
    r = np.sqrt(p0x**2 + p0y**2)
    likelihood = (
        stats.norm.pdf(r - 0.1, scale=0.01)
        * stats.uniform.pdf(a, loc=-np.pi/2, scale=np.pi)
    )
    return likelihood

num_dim = 2
x_o = torch.zeros(num_dim)

## Ground truth

In [ ]:
xx, yy = np.meshgrid(
    *[np.linspace(-.5, .5, 200)] * 2
)
thetas = np.stack([xx, yy], axis=-1)

zz = np.vectorize(two_moons_likelihood, signature='(n),(n)->()')(thetas, x_o)

In [ ]:
plt.figure()
plt.contourf(xx, yy, zz)
plt.gca().set_aspect('equal')

## SBI

In [ ]:
# Check prior, return PyTorch prior.
prior = BoxUniform(low=-2 * torch.ones(num_dim), high=2 * torch.ones(num_dim))
prior, num_parameters, prior_returns_numpy = process_prior(prior)

# Check simulator, returns PyTorch simulator able to simulate batches.
simulator = process_simulator(two_moons, prior, prior_returns_numpy)

# Consistency check after making ready for sbi.
check_sbi_inputs(simulator, prior)

In [ ]:
def run_inference(inference,
                  num_rounds=10,
                  num_simulations_per_round=1000):
    proposal = prior

    for i in range(num_rounds):
        theta, x = simulate_for_sbi(simulator, proposal,
                                    num_simulations=num_simulations_per_round)
        inference.append_simulations(theta, x, proposal=proposal)
        density_estimator = inference.train()
        posterior = inference.build_posterior(density_estimator)
        proposal = posterior.set_default_x(x_o)

        plot_samples(posterior, f'Round {i+1}')

    return posterior


def plot_samples(posterior, title=None, show_ground_truth=True):
    posterior_samples = posterior.sample((10000,), x=x_o)

    plt.figure()
    plt.hist2d(*posterior_samples.T, bins=200, range=[(-1, 1), (-1, 1)])
    if show_ground_truth:
        plt.contour(xx, yy, zz, colors='w', levels=[zz.max() / 2],
                    linestyles='--', linewidths=1, alpha=.7)
    plt.gca().set_aspect('equal')
    plt.title(title)


In [ ]:
inference_d = NPE_D(prior=prior, density_estimator='nsf')
posterior_d = run_inference(inference_d)

In [ ]:
inference_c = NPE_C(prior=prior, density_estimator='nsf')
posterior_c = run_inference(inference_c)

In [ ]:
inference_b = NPE_B(prior=prior, density_estimator='nsf')
posterior_b = run_inference(inference_b)

In [ ]:
plot_samples(posterior_b, show_ground_truth=False)

In [ ]:
plot_samples(posterior_c, show_ground_truth=False)

In [ ]:
plot_samples(posterior_d, show_ground_truth=False)